# Hari 008a — Model Ising 2D Lintas Tiga Bahasa (C, C++, Python)
**Metropolis, Wolff, Swendsen–Wang, Finite-Size Scaling, dan Paritas Bit-Identik**

Notebook ini merupakan bagian Python dari proyek tiga bahasa [`projects/c/day008a_ising_trilingual`](../projects/c/day008a_ising_trilingual/README.md) sesuai kontrak [`SPEC.md`](../projects/c/day008a_ising_trilingual/SPEC.md).

In [1]:
from __future__ import annotations
import csv, hashlib, json, math, os, platform, subprocess, time
import xml.etree.ElementTree as ET
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

T_NB_START = time.perf_counter()
ROOT = Path.cwd() if (Path.cwd() / "days").exists() else Path.cwd().parent
PROJ_DIR = ROOT / "projects" / "c" / "day008a_ising_trilingual"
DATA_DIR = PROJ_DIR / "data"
OUT_DIR = ROOT / "outputs" / "day008a"
FIG_DIR = ROOT / "outputs" / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

MASTER_SEED = 20261126
TC_EXACT = 2.0 / math.log(1.0 + math.sqrt(2.0))
E_TC_EXACT = -math.sqrt(2.0)

CRITERIA_APRIORI = {
    "c1_parity_bit_identical": "0 selisih pada 3000 langkah + hash FNV-1a akhir identik di C, C++, Python untuk 3 algoritma",
    "c2_exact_enumeration": "Enumerasi L=4 (3 bahasa) & L=5 (C/C++) selisih relatif <= 1e-12; simulasi L=4 cocok <= 4 SE",
    "c3_thread_invariance": "Hasil simulasi C dan C++ bit-identik untuk 1, 2, 4, 8 thread",
    "c4_onsager_energy": "Energi L=128 di T in {2.0, 2.6, 3.2} cocok Onsager dalam <= 4 SE (+ bias ukuran-hingga)",
    "c5_yang_magnetization": "|m| di T=2.0 (L=128) & T=2.1 (L=64) cocok Yang dalam <= 4 SE (+ bias ukuran-hingga)",
    "c6_binder_crossing": "Titik silang Binder U4(L, 2L) dalam |T - T_c| <= 0.01 untuk L >= 16",
    "c7_gamma_over_nu": "|gamma_hat/nu_hat - 1.75| <= 0.10",
    "c8_beta_over_nu": "|beta_hat/nu_hat - 0.125| <= 0.03",
    "c9_dynamic_exponent": "z_Wolff < z_Metropolis - 1.0 dan z_Metropolis in [1.8, 2.5]",
    "c10_notebook_reproducible": "Notebook berjalan bersih dua kali dengan angka kunci identik",
    "c11_all_tests_pass": "make test C, make test C++, dan sel assert notebook lulus",
    "c12_figures_valid": "8 pasang grafik PNG+SVG di outputs/figures/day008a_* ada, tak kosong, dan SVG valid XML",
    "c13_code_size_reported": "Ukuran kode dilaporkan dan tiap berkas sumber <= 600 baris",
    "c14_speed_comparison": "Perbandingan kecepatan 3 bahasa (ns/spin-update, median+-MAD >= 7 ulang) dilaporkan terukur",
}
print(f"Python {platform.python_version()}, arch={platform.machine()}, TC_EXACT={TC_EXACT:.15f}")


Python 3.11.18, arch=arm64, TC_EXACT=2.269185314213022


## 1. RNG Deterministik (`SplitMix64` + `xoshiro256++`) & Hash FNV-1a 64-bit

In [2]:
MASK64 = 0xFFFFFFFFFFFFFFFF

def splitmix64_mix(z: int) -> int:
    z = ((z ^ (z >> 30)) * 0xBF58476D1CE4E5B9) & MASK64
    z = ((z ^ (z >> 27)) * 0x94D049BB133111EB) & MASK64
    return (z ^ (z >> 31)) & MASK64

class Xoshiro256pp:
    def __init__(self, seed: int = MASTER_SEED):
        st = seed & MASK64
        self.s = [0, 0, 0, 0]
        for i in range(4):
            st = (st + 0x9E3779B97F4A7C15) & MASK64
            self.s[i] = splitmix64_mix(st)

    def next_u64(self) -> int:
        s = self.s
        x = (s[0] + s[3]) & MASK64
        rot = ((x << 23) | (x >> 41)) & MASK64
        res = (rot + s[0]) & MASK64
        t = (s[1] << 17) & MASK64
        s[2] ^= s[0]
        s[3] ^= s[1]
        s[1] ^= s[2]
        s[0] ^= s[3]
        s[2] ^= t
        s[3] = ((s[3] << 45) | (s[3] >> 19)) & MASK64
        return res

    def u01(self) -> float:
        return (self.next_u64() >> 11) * (2.0 ** -53)

    def rand_int(self, n: int) -> int:
        idx = int(math.floor(self.u01() * n))
        return n - 1 if idx >= n else idx

def compute_task_seed(L: int, t_index: int, algo_id: int, replica: int) -> int:
    h = 0x517CC1B727220A95
    for v in (L, t_index, algo_id, replica):
        h = splitmix64_mix(h ^ (v & MASK64))
    return splitmix64_mix(MASTER_SEED ^ h)

def fnv1a_64(spins: list[int]) -> int:
    h = 0xCBF29CE484222325
    for s in spins:
        h ^= (s & 0xFF)
        h = (h * 0x100000001B3) & MASK64
    return h

# Sel Assert RNG & Hash
_rng_t = Xoshiro256pp(MASTER_SEED)
assert _rng_t.next_u64() == 0xA9920E0E7DAB3EF0
assert _rng_t.next_u64() == 0x033B9CEB4B3BA7A0
assert fnv1a_64([1, -1, 1, 1]) == fnv1a_64([1, -1, 1, 1])
print("[ASSERT OK] RNG xoshiro256++ & FNV-1a cocok vektor acuan")


[ASSERT OK] RNG xoshiro256++ & FNV-1a cocok vektor acuan


## 2. Implementasi Model Ising 2D (Metropolis, Wolff, Swendsen–Wang)

In [3]:
class IsingPython:
    def __init__(self, L: int):
        self.L = L
        self.N = L * L
        self.spins = [1] * self.N
        self.nbr = [0] * (self.N * 4)
        for y in range(L):
            yp = 0 if y + 1 == L else y + 1
            ym = L - 1 if y == 0 else y - 1
            for x in range(L):
                xp = 0 if x + 1 == L else x + 1
                xm = L - 1 if x == 0 else x - 1
                i = y * L + x
                b = i * 4
                self.nbr[b] = y * L + xp
                self.nbr[b + 1] = yp * L + x
                self.nbr[b + 2] = y * L + xm
                self.nbr[b + 3] = ym * L + x

    def set_cold(self):
        self.spins = [1] * self.N

    def set_hot(self, rng: Xoshiro256pp):
        self.spins = [1 if rng.u01() < 0.5 else -1 for _ in range(self.N)]

    def energy_mag(self) -> tuple[int, int]:
        E = 0
        M = 0
        s = self.spins
        nbr = self.nbr
        for i in range(self.N):
            si = s[i]
            M += si
            b = i * 4
            E -= si * (s[nbr[b]] + s[nbr[b + 1]])
        return E, M

    def metropolis_sweep(self, rng: Xoshiro256pp, p4: float, p8: float):
        s = self.spins
        nbr = self.nbr
        u01 = rng.u01
        for i in range(self.N):
            b = i * 4
            sum_nn = s[nbr[b]] + s[nbr[b + 1]] + s[nbr[b + 2]] + s[nbr[b + 3]]
            dE = 2 * s[i] * sum_nn
            u = u01()
            if dE <= 0 or (dE == 4 and u < p4) or (dE == 8 and u < p8):
                s[i] = -s[i]

    def wolff_step(self, rng: Xoshiro256pp, p_add: float):
        s = self.spins
        nbr = self.nbr
        u01 = rng.u01
        i0 = rng.rand_int(self.N)
        old_s = s[i0]
        new_s = -old_s
        s[i0] = new_s
        stack = [i0]
        while stack:
            curr = stack.pop()
            b = curr * 4
            for d in range(4):
                nb = nbr[b + d]
                if s[nb] == old_s and u01() < p_add:
                    s[nb] = new_s
                    stack.append(nb)

    def sw_step(self, rng: Xoshiro256pp, p_add: float):
        N = self.N
        s = self.spins
        nbr = self.nbr
        u01 = rng.u01
        parent = [-1] * N

        def find(x: int) -> int:
            while parent[x] >= 0:
                px = parent[x]
                if parent[px] >= 0:
                    parent[x] = parent[px]
                x = px
            return x

        def unite(a: int, b: int):
            ra, rb = find(a), find(b)
            if ra == rb:
                return
            if -parent[ra] < -parent[rb]:
                ra, rb = rb, ra
            parent[ra] += parent[rb]
            parent[rb] = ra

        for i in range(N):
            si = s[i]
            r_nb = nbr[i * 4]
            if si == s[r_nb] and u01() < p_add:
                unite(i, r_nb)
            u_nb = nbr[i * 4 + 1]
            if si == s[u_nb] and u01() < p_add:
                unite(i, u_nb)

        flip = [0] * N
        for i in range(N):
            if parent[i] < 0 and u01() < 0.5:
                flip[i] = 1
        for i in range(N):
            if flip[find(i)]:
                s[i] = -s[i]


## 3. Uji Paritas Bit-Identik 3 Bahasa (`c1`)

In [4]:
PARITY_RESULTS = {}
algos = [(0, "metropolis"), (1, "wolff"), (2, "sw")]
beta_par = 0.44
p4_par, p8_par = math.exp(-4.0 * beta_par), math.exp(-8.0 * beta_par)
padd_par = 1.0 - math.exp(-2.0 * beta_par)

def read_parity_csv(path: Path):
    rows = []
    with path.open() as f:
        r = csv.DictReader(f)
        for row in r:
            rows.append((int(row["E_sum"]), int(row["M_sum"])))
    return rows

def read_parity_summary(path: Path):
    out = {}
    with path.open() as f:
        for row in csv.DictReader(f):
            out[row["algo"]] = row
    return out

sum_c = read_parity_summary(DATA_DIR / "parity_c_summary.csv")
sum_cpp = read_parity_summary(DATA_DIR / "parity_cpp_summary.csv")

all_parity_ok = True
for aid, aname in algos:
    lat = IsingPython(8)
    lat.set_cold()
    rng = Xoshiro256pp(compute_task_seed(8, 0, aid, 0))
    py_series = []
    for _ in range(3000):
        if aid == 0:
            lat.metropolis_sweep(rng, p4_par, p8_par)
        elif aid == 1:
            lat.wolff_step(rng, padd_par)
        else:
            lat.sw_step(rng, padd_par)
        py_series.append(lat.energy_mag())
    py_hash = f"{fnv1a_64(lat.spins):016x}"
    c_series = read_parity_csv(DATA_DIR / f"parity_c_{aname}.csv")
    cpp_series = read_parity_csv(DATA_DIR / f"parity_cpp_{aname}.csv")
    diff_c_cpp = sum(1 for a, b in zip(c_series, cpp_series) if a != b)
    diff_c_py = sum(1 for a, b in zip(c_series, py_series) if a != b)
    diff_cpp_py = sum(1 for a, b in zip(cpp_series, py_series) if a != b)
    hash_match = (sum_c[aname]["final_fnv1a_hex"] == sum_cpp[aname]["final_fnv1a_hex"] == py_hash)
    PARITY_RESULTS[aname] = {
        "steps": 3000,
        "diff_c_cpp": diff_c_cpp,
        "diff_c_py": diff_c_py,
        "diff_cpp_py": diff_cpp_py,
        "c_hash": sum_c[aname]["final_fnv1a_hex"],
        "cpp_hash": sum_cpp[aname]["final_fnv1a_hex"],
        "py_hash": py_hash,
        "bit_identical": (diff_c_cpp == 0 and diff_c_py == 0 and diff_cpp_py == 0 and hash_match),
    }
    all_parity_ok = all_parity_ok and PARITY_RESULTS[aname]["bit_identical"]
    print(f"[c1] {aname:10s}: diffs=(C-C++:{diff_c_cpp}, C-Py:{diff_c_py}, C++-Py:{diff_cpp_py}), hash={py_hash}, match={hash_match}")

assert all_parity_ok, "Pelanggaran c1: ketidakcocokan paritas bit-identik!"


[c1] metropolis: diffs=(C-C++:0, C-Py:0, C++-Py:0), hash=7458071e0380ee91, match=True


[c1] wolff     : diffs=(C-C++:0, C-Py:0, C++-Py:0), hash=52911df0a1ba8509, match=True


[c1] sw        : diffs=(C-C++:0, C-Py:0, C++-Py:0), hash=50d1e256b6df3e61, match=True


## 4. Anchor Eksak: Enumerasi $L=4, 5$, Integral Onsager (AGM), dan Magnetisasi Yang (`c2`)

In [5]:
def enumerate_exact_py(L: int, beta: float):
    N = L * L
    n_rows = 1 << L
    row_mask = n_rows - 1
    e_row, m_row = [0] * n_rows, [0] * n_rows
    for r in range(n_rows):
        er, mr = 0, 0
        for x in range(L):
            s0 = 1 if ((r >> x) & 1) else -1
            s1 = 1 if ((r >> ((x + 1) % L)) & 1) else -1
            mr += s0
            er -= s0 * s1
        e_row[r], m_row[r] = er, mr
    e_vert = [[0] * n_rows for _ in range(n_rows)]
    for r1 in range(n_rows):
        for r2 in range(n_rows):
            e_vert[r1][r2] = -sum(
                (1 if ((r1 >> x) & 1) else -1) * (1 if ((r2 >> x) & 1) else -1)
                for x in range(L)
            )
    w_E, m2_E = {}, {}
    for mask in range(1 << N):
        rs = [(mask >> (y * L)) & row_mask for y in range(L)]
        E = sum(e_row[ry] for ry in rs) + sum(e_vert[rs[y]][rs[(y + 1) % L]] for y in range(L))
        M = sum(m_row[ry] for ry in rs)
        w_E[E] = w_E.get(E, 0.0) + 1.0
        m2_E[E] = m2_E.get(E, 0.0) + M * M
    Z = sum(w * math.exp(-beta * E) for E, w in w_E.items())
    sumE = sum(E * w * math.exp(-beta * E) for E, w in w_E.items())
    sumM2 = sum(m2_E[E] * math.exp(-beta * E) for E in w_E)
    return {"Z": Z, "e_mean": (sumE / Z) / N, "m2_mean": (sumM2 / Z) / (N * N)}

def onsager_energy_agm(beta: float) -> float:
    s2, c2, t2 = math.sinh(2.0 * beta), math.cosh(2.0 * beta), math.tanh(2.0 * beta)
    k1 = (2.0 * s2) / (c2 * c2)
    kp = math.sqrt(max(0.0, 1.0 - k1 * k1))
    coth2b = c2 / s2
    if kp < 1e-14:
        return -coth2b
    a, b = 1.0, kp
    for _ in range(30):
        a, b = 0.5 * (a + b), math.sqrt(a * b)
        if abs(a - b) < 1e-16:
            break
    K_k1 = math.pi / (2.0 * a)
    return -coth2b * (1.0 + (2.0 / math.pi) * (2.0 * t2 * t2 - 1.0) * K_k1)

def yang_mag(beta: float) -> float:
    val = 1.0 - math.sinh(2.0 * beta) ** -4
    return val ** 0.125 if val > 0.0 else 0.0

assert abs(onsager_energy_agm(1.0 / TC_EXACT) - E_TC_EXACT) < 1e-12
r_l3 = enumerate_exact_py(3, 0.0)
assert abs(r_l3["Z"] - 512.0) < 1e-12 and abs(r_l3["m2_mean"] - 1.0 / 9.0) < 1e-12

def read_enum_csv(path: Path):
    out = {}
    with path.open() as f:
        for row in csv.DictReader(f):
            out[(int(row["L"]), round(float(row["beta"]), 12))] = {
                "Z": float(row["Z"]), "e_mean": float(row["e_mean"]), "m2_mean": float(row["m2_mean"])
            }
    return out

enum_c = read_enum_csv(DATA_DIR / "enum_c.csv")
enum_cpp = read_enum_csv(DATA_DIR / "enum_cpp.csv")
betas_enum = [0.2, 0.35, 0.4406867935097715, 0.5, 0.7]
max_enum_rel_err = 0.0
for b in betas_enum:
    py4 = enumerate_exact_py(4, b)
    c4 = enum_c[(4, round(b, 12))]
    cpp4 = enum_cpp[(4, round(b, 12))]
    c5 = enum_c[(5, round(b, 12))]
    cpp5 = enum_cpp[(5, round(b, 12))]
    for key in ("Z", "e_mean", "m2_mean"):
        max_enum_rel_err = max(
            max_enum_rel_err,
            abs(py4[key] - c4[key]) / max(1e-15, abs(c4[key])),
            abs(py4[key] - cpp4[key]) / max(1e-15, abs(cpp4[key])),
            abs(c5[key] - cpp5[key]) / max(1e-15, abs(c5[key])),
        )
assert max_enum_rel_err <= 1e-12, f"Enum rel err {max_enum_rel_err}"

# Uji simulasi L=4 Wolff cocok eksak <= 4 SE
lat4 = IsingPython(4)
rng4 = Xoshiro256pp(compute_task_seed(4, 99, 1, 0))
lat4.set_hot(rng4)
b_test = 0.4406867935097715
padd4 = 1.0 - math.exp(-2.0 * b_test)
for _ in range(500):
    lat4.wolff_step(rng4, padd4)
e_sim4 = []
for _ in range(10000):
    lat4.wolff_step(rng4, padd4)
    Ei, _ = lat4.energy_mag()
    e_sim4.append(Ei / 16.0)
e_sim4_arr = np.array(e_sim4)
e_sim4_mean = float(np.mean(e_sim4_arr))
e_sim4_se = float(np.std(e_sim4_arr, ddof=1) / math.sqrt(len(e_sim4_arr) / 4.0))
z_l4_sim = abs(e_sim4_mean - enum_c[(4, round(b_test, 12))]["e_mean"]) / e_sim4_se
assert z_l4_sim <= 4.0, f"Simulasi L=4 meleset > 4 SE: {z_l4_sim}"
print(f"[c2] Max rel err enumerasi L=4,5 = {max_enum_rel_err:.2e} <= 1e-12; L=4 sim Z={z_l4_sim:.2f} <= 4.0")


[c2] Max rel err enumerasi L=4,5 = 2.92e-16 <= 1e-12; L=4 sim Z=0.80 <= 4.0


## 5. Beban Python (Skala Terkurangi) & Benchmark Kecepatan Tiga Bahasa (`c14`)

In [6]:
# Beban Python: L in {8, 16, 32} penuh (36 T), L=64 dikurangi (12 T rapat), L=128 tidak dijalankan
TEMP_GRID = sorted(set(
    [1.8 + (3.4 - 1.8) * i / 24.0 for i in range(25)] +
    [2.15 + (2.40 - 2.15) * i / 11.0 for i in range(12)]
))
assert len(TEMP_GRID) == 36

t0_py_sim = time.perf_counter()
PY_SIM_ROWS = []
for L, n_therm, n_meas, t_indices in [
    (8, 150, 600, list(range(36))),
    (16, 150, 500, list(range(36))),
    (32, 100, 300, list(range(36))),
    (64, 60, 150, [i for i, T in enumerate(TEMP_GRID) if 2.15 <= T <= 2.40]),
]:
    lat = IsingPython(L)
    invN = 1.0 / (L * L)
    for ti in t_indices:
        T = TEMP_GRID[ti]
        beta = 1.0 / T
        p_add = 1.0 - math.exp(-2.0 * beta)
        rng = Xoshiro256pp(compute_task_seed(L, ti, 1, 0))
        lat.set_hot(rng)
        for _ in range(n_therm):
            lat.wolff_step(rng, p_add)
        es, ams = [], []
        for _ in range(n_meas):
            lat.wolff_step(rng, p_add)
            Ei, Mi = lat.energy_mag()
            es.append(Ei * invN)
            ams.append(abs(Mi) * invN)
        PY_SIM_ROWS.append({
            "L": L, "T": T, "n_therm": n_therm, "n_meas": n_meas,
            "e_mean": float(np.mean(es)), "abs_m_mean": float(np.mean(ams)),
        })
T_PY_SIM_SEC = time.perf_counter() - t0_py_sim
print(f"Simulasi Wolff Python ({len(PY_SIM_ROWS)} titik L=8..64) selesai dalam {T_PY_SIM_SEC:.2f}s")

# Benchmark Python ns/spin-update (7 ulang di L=32, T=Tc)
def median_mad(arr):
    a = np.asarray(arr, dtype=float)
    med = float(np.median(a))
    mad = float(np.median(np.abs(a - med)))
    return med, mad

py_bench = {"metropolis": [], "wolff": [], "sw": []}
p4_tc, p8_tc = math.exp(-4.0 / TC_EXACT), math.exp(-8.0 / TC_EXACT)
padd_tc = 1.0 - math.exp(-2.0 / TC_EXACT)
for rep in range(7):
    lat = IsingPython(32)
    rng = Xoshiro256pp(compute_task_seed(32, 0, 0, rep))
    lat.set_hot(rng)
    t0 = time.perf_counter()
    for _ in range(20):
        lat.metropolis_sweep(rng, p4_tc, p8_tc)
    py_bench["metropolis"].append((time.perf_counter() - t0) * 1e9 / (20 * 1024))

    lat.set_hot(rng)
    t0 = time.perf_counter()
    for _ in range(40):
        lat.wolff_step(rng, padd_tc)
    py_bench["wolff"].append((time.perf_counter() - t0) * 1e9 / (40 * 1024))

    lat.set_hot(rng)
    t0 = time.perf_counter()
    for _ in range(15):
        lat.sw_step(rng, padd_tc)
    py_bench["sw"].append((time.perf_counter() - t0) * 1e9 / (15 * 1024))

BENCH_SUMMARY = {}
for lang_file in ("bench_c.csv", "bench_cpp.csv"):
    by_var = {}
    with (DATA_DIR / lang_file).open() as f:
        for row in csv.DictReader(f):
            key = f"{row['lang']}_{row['variant']}"
            by_var.setdefault(key, []).append(float(row["ns_per_spin_update"]))
    for k, vals in by_var.items():
        m, d = median_mad(vals)
        BENCH_SUMMARY[k] = {"median_ns": round(m, 3), "mad_ns": round(d, 3)}
for k, vals in py_bench.items():
    m, d = median_mad(vals)
    BENCH_SUMMARY[f"python_{k}"] = {"median_ns": round(m, 3), "mad_ns": round(d, 3)}

for k, v in BENCH_SUMMARY.items():
    print(f"  {k:28s}: {v['median_ns']:9.3f} +- {v['mad_ns']:.3f} ns/spin-update")


Simulasi Wolff Python (123 titik L=8..64) selesai dalam 13.75s


  c_metropolis_table          :     6.603 +- 0.057 ns/spin-update
  c_metropolis_computed       :     3.476 +- 0.015 ns/spin-update
  c_wolff                     :     7.856 +- 0.173 ns/spin-update
  c_sw                        :    19.752 +- 0.099 ns/spin-update
  cpp_metropolis_table        :     6.524 +- 0.011 ns/spin-update
  cpp_metropolis_computed     :     4.799 +- 0.028 ns/spin-update
  cpp_metropolis_constexpr32  :     4.588 +- 0.011 ns/spin-update
  cpp_wolff                   :     8.331 +- 0.144 ns/spin-update
  cpp_sw                      :    18.845 +- 0.079 ns/spin-update
  python_metropolis           :   867.063 +- 106.049 ns/spin-update
  python_wolff                :    20.352 +- 5.787 ns/spin-update
  python_sw                   :  1661.784 +- 101.074 ns/spin-update


## 6. Analisis Fisika Statistik & Finite-Size Scaling (dari CSV C/C++)

In [7]:
def load_full_csv(path: Path):
    rows = []
    with path.open() as f:
        for r in csv.DictReader(f):
            row = {k: (int(r[k]) if k in ("L", "t_idx", "n_therm", "n_meas")
                       else (r[k] if k in ("algo", "final_hash") else float(r[k])))
                   for k in r}
            rows.append(row)
    return rows

FULL_C = load_full_csv(DATA_DIR / "full_c.csv")
FULL_CPP = load_full_csv(DATA_DIR / "full_cpp.csv")

# Cek c3: invarian thread 1, 2, 4, 8
def read_lines(p: Path):
    return p.read_text(encoding="utf-8")

c_t1 = read_lines(DATA_DIR / "threads_c_t1.csv")
cpp_t1 = read_lines(DATA_DIR / "threads_cpp_t1.csv")
C3_THREADS_OK = all(
    read_lines(DATA_DIR / f"threads_c_t{nt}.csv") == c_t1 and
    read_lines(DATA_DIR / f"threads_cpp_t{nt}.csv") == cpp_t1
    for nt in (1, 2, 4, 8)
) and (c_t1 == cpp_t1)
assert C3_THREADS_OK, "Pelanggaran c3: hasil thread tidak identik!"

# Gunakan FULL_C (identik dengan C++) untuk analisis fisika
by_key = {(r["algo"], r["L"], r["t_idx"]): r for r in FULL_C}
wolff_by_L = {L: sorted([r for r in FULL_C if r["algo"] == "wolff" and r["L"] == L], key=lambda x: x["T"])
              for L in (8, 16, 32, 64, 128)}

# c4: Onsager pada L=128 di T in {2.0, 2.6, 3.2}
C4_ONSAGER = {}
for T_target in (2.0, 2.6, 3.2):
    r = min(wolff_by_L[128], key=lambda x: abs(x["T"] - T_target))
    e_exact = onsager_energy_agm(1.0 / r["T"])
    z_val = abs(r["e_mean"] - e_exact) / max(1e-12, r["e_err"])
    C4_ONSAGER[str(T_target)] = {
        "T": round(r["T"], 6), "e_sim": round(r["e_mean"], 6), "e_err": round(r["e_err"], 6),
        "e_onsager": round(e_exact, 6), "z_score": round(z_val, 3), "within_4se": bool(z_val <= 4.0)
    }
    print(f"[c4] Onsager T={r['T']:.4f}: sim={r['e_mean']:.6f}+-{r['e_err']:.6f}, exact={e_exact:.6f}, Z={z_val:.2f}")

# Ekstrapolasi 1/L^2 di Tc
idx_tc = min(range(len(TEMP_GRID)), key=lambda i: abs(TEMP_GRID[i] - TC_EXACT))
T_near_tc = TEMP_GRID[idx_tc]
e_tc_L = [by_key[("wolff", L, idx_tc)]["e_mean"] for L in (16, 32, 64, 128)]
invL2 = np.array([1.0 / (L * L) for L in (16, 32, 64, 128)])
slope_e_tc, intercept_e_tc = np.polyfit(invL2, e_tc_L, 1)
print(f"[Tc ekstrapolasi] e(L->inf) = {intercept_e_tc:.6f} vs -sqrt(2) = {E_TC_EXACT:.6f}")

# c5: Yang spontaneous magnetization
r_y20 = min(wolff_by_L[128], key=lambda x: abs(x["T"] - 2.0))
r_y21 = min(wolff_by_L[64], key=lambda x: abs(x["T"] - 2.1))
C5_YANG = {}
for lbl, r in [("T2.0_L128", r_y20), ("T2.1_L64", r_y21)]:
    y_ex = yang_mag(1.0 / r["T"])
    z_val = abs(r["abs_m_mean"] - y_ex) / max(1e-12, r["abs_m_err"])
    C5_YANG[lbl] = {
        "T": round(r["T"], 6), "abs_m_sim": round(r["abs_m_mean"], 6), "abs_m_err": round(r["abs_m_err"], 6),
        "m_yang": round(y_ex, 6), "z_score": round(z_val, 3), "within_4se": bool(z_val <= 4.0)
    }
    print(f"[c5] Yang {lbl}: sim={r['abs_m_mean']:.6f}+-{r['abs_m_err']:.6f}, exact={y_ex:.6f}, Z={z_val:.2f}")

# c6: Binder cumulant crossings (L, 2L)
BINDER_CROSSINGS = {}
for L1, L2 in [(8, 16), (16, 32), (32, 64), (64, 128)]:
    Ts = np.array([r["T"] for r in wolff_by_L[L1]])
    diff = np.array([r["binder"] for r in wolff_by_L[L1]]) - np.array([r["binder"] for r in wolff_by_L[L2]])
    # Cari perpotongan terdekat dari Tc
    cross_t = None
    best_dist = 1e9
    for i in range(len(Ts) - 1):
        if diff[i] * diff[i + 1] <= 0 and diff[i] != diff[i + 1]:
            t_c = Ts[i] - diff[i] * (Ts[i + 1] - Ts[i]) / (diff[i + 1] - diff[i])
            if abs(t_c - TC_EXACT) < best_dist:
                best_dist = abs(t_c - TC_EXACT)
                cross_t = float(t_c)
    BINDER_CROSSINGS[f"{L1}_{L2}"] = {
        "T_cross": round(cross_t, 6) if cross_t else None,
        "abs_err": round(abs(cross_t - TC_EXACT), 6) if cross_t else None,
        "within_0.01": bool(cross_t is not None and abs(cross_t - TC_EXACT) <= 0.01),
    }
    print(f"[c6] Binder ({L1},{L2}): T_cross={cross_t:.6f}, |dT|={abs(cross_t - TC_EXACT):.6f}")

# c7: chi_max ~ L^(gamma/nu)
Ls_fit = np.array([8, 16, 32, 64, 128], dtype=float)
chi_peaks = np.array([max(r["chi"] for r in wolff_by_L[int(L)]) for L in Ls_fit])
gamma_nu_hat, c_chi = np.polyfit(np.log(Ls_fit), np.log(chi_peaks), 1)
res_chi = np.log(chi_peaks) - (gamma_nu_hat * np.log(Ls_fit) + c_chi)
se_gamma_nu = float(np.sqrt(np.sum(res_chi**2) / (len(Ls_fit) - 2) / np.sum((np.log(Ls_fit) - np.mean(np.log(Ls_fit)))**2)))

# c8: |m|(Tc) ~ L^(-beta/nu)
m_tc = np.array([by_key[("wolff", int(L), idx_tc)]["abs_m_mean"] for L in Ls_fit])
slope_m, c_m = np.polyfit(np.log(Ls_fit), np.log(m_tc), 1)
beta_nu_hat = float(-slope_m)
res_m = np.log(m_tc) - (slope_m * np.log(Ls_fit) + c_m)
se_beta_nu = float(np.sqrt(np.sum(res_m**2) / (len(Ls_fit) - 2) / np.sum((np.log(Ls_fit) - np.mean(np.log(Ls_fit)))**2)))

# c9: tau_int vs L di Tc untuk 3 algoritma
Z_DYN = {}
for algo_nm in ("metropolis", "wolff", "sw"):
    taus = np.array([by_key[(algo_nm, int(L), idx_tc)]["tau_m"] for L in Ls_fit])
    z_hat, c_z = np.polyfit(np.log(Ls_fit), np.log(taus), 1)
    res_z = np.log(taus) - (z_hat * np.log(Ls_fit) + c_z)
    se_z = float(np.sqrt(np.sum(res_z**2) / (len(Ls_fit) - 2) / np.sum((np.log(Ls_fit) - np.mean(np.log(Ls_fit)))**2)))
    Z_DYN[algo_nm] = {"z_hat": round(float(z_hat), 4), "se": round(se_z, 4), "taus": [round(float(t), 3) for t in taus]}
    print(f"[c9] {algo_nm:10s}: z = {z_hat:.4f} +- {se_z:.4f}, taus={Z_DYN[algo_nm]['taus']}")

print(f"[c7] gamma/nu = {gamma_nu_hat:.4f} +- {se_gamma_nu:.4f} (teori 1.75)")
print(f"[c8] beta/nu  = {beta_nu_hat:.4f} +- {se_beta_nu:.4f} (teori 0.125)")


[c4] Onsager T=2.0000: sim=-1.745577+-0.000106, exact=-1.745565, Z=0.12
[c4] Onsager T=2.6000: sim=-1.026033+-0.004961, exact=-1.028294, Z=0.46
[c4] Onsager T=3.2000: sim=-0.737463+-0.008128, exact=-0.745148, Z=0.95
[Tc ekstrapolasi] e(L->inf) = -1.426869 vs -sqrt(2) = -1.414214
[c5] Yang T2.0_L128: sim=0.911323+-0.000053, exact=0.911319, Z=0.07
[c5] Yang T2.1_L64: sim=0.885413+-0.000077, exact=0.885490, Z=0.99
[c6] Binder (8,16): T_cross=2.264444, |dT|=0.004741
[c6] Binder (16,32): T_cross=2.266860, |dT|=0.002325
[c6] Binder (32,64): T_cross=2.268691, |dT|=0.000494
[c6] Binder (64,128): T_cross=2.267882, |dT|=0.001303
[c9] metropolis: z = 1.8625 +- 0.1105, taus=[3.01, 10.214, 43.799, 205.63, 426.52]
[c9] wolff     : z = 0.4118 +- 0.0191, taus=[1.558, 2.215, 2.94, 3.926, 4.878]
[c9] sw        : z = 0.1998 +- 0.0258, taus=[2.479, 2.906, 3.562, 4.054, 4.195]
[c7] gamma/nu = 1.7682 +- 0.0061 (teori 1.75)
[c8] beta/nu  = 0.1122 +- 0.0030 (teori 0.125)


## 7. Visualisasi Wajib (`outputs/figures/day008a_*` PNG + SVG)

In [8]:
SAVED_FIGS = {}

def save_fig(fig, name: str):
    png_p = FIG_DIR / f"day008a_{name}.png"
    svg_p = FIG_DIR / f"day008a_{name}.svg"
    fig.savefig(png_p, dpi=150, bbox_inches="tight")
    fig.savefig(svg_p, bbox_inches="tight")
    plt.close(fig)
    SAVED_FIGS[name] = [f"outputs/figures/day008a_{name}.png", f"outputs/figures/day008a_{name}.svg"]

# 1. Snapshot spin L=128 (T=2.0, Tc, 3.0) dari C
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (lbl, title) in zip(axes, [("T2.0", "T = 2.0 (< Tc)"), ("Tc", "T = Tc = 2.269"), ("T3.0", "T = 3.0 (> Tc)")]):
    mat = np.loadtxt(DATA_DIR / f"snapshot_L128_{lbl}.csv", delimiter=",")
    ax.imshow(mat, cmap="coolwarm", interpolation="nearest", vmin=-1, vmax=1)
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Snapshot Spin Ising 2D L=128 (C11, Wolff)", fontsize=12)
save_fig(fig, "snapshots")

# 2. E(T) & |m|(T) vs Onsager / Yang
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
Ts_fine = np.linspace(1.8, 3.4, 200)
ax1.plot(Ts_fine, [onsager_energy_agm(1.0 / T) for T in Ts_fine], "k--", lw=1.5, label="Onsager (L->inf)")
ax2.plot(Ts_fine, [yang_mag(1.0 / T) if T < TC_EXACT else 0.0 for T in Ts_fine], "k--", lw=1.5, label="Yang (L->inf)")
for L in (16, 32, 64, 128):
    rs = wolff_by_L[L]
    Ts = [r["T"] for r in rs]
    ax1.errorbar(Ts, [r["e_mean"] for r in rs], yerr=[r["e_err"] for r in rs], fmt="o-", ms=3, lw=1, label=f"L={L}")
    ax2.errorbar(Ts, [r["abs_m_mean"] for r in rs], yerr=[r["abs_m_err"] for r in rs], fmt="o-", ms=3, lw=1, label=f"L={L}")
ax1.axvline(TC_EXACT, color="gray", ls=":", lw=1); ax2.axvline(TC_EXACT, color="gray", ls=":", lw=1)
ax1.set_xlabel("T"); ax1.set_ylabel("Energi per situs e"); ax1.set_title("e(T) vs Onsager"); ax1.legend(fontsize=8)
ax2.set_xlabel("T"); ax2.set_ylabel("|m|"); ax2.set_title("|m|(T) vs Yang"); ax2.legend(fontsize=8)
save_fig(fig, "energy_mag")

# 3. Binder crossing
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for L in (8, 16, 32, 64, 128):
    rs = [r for r in wolff_by_L[L] if 2.1 <= r["T"] <= 2.45]
    ax.plot([r["T"] for r in rs], [r["binder"] for r in rs], "o-", ms=3, label=f"L={L}")
ax.axvline(TC_EXACT, color="k", ls="--", lw=1.2, label=f"Tc={TC_EXACT:.4f}")
ax.axhline(0.6107, color="gray", ls=":", lw=1, label="U4* ~ 0.6107 (lit.)")
ax.set_xlabel("T"); ax.set_ylabel("Binder Cumulant U4"); ax.set_title("Titik Silang Binder U4(T, L)"); ax.legend(fontsize=8)
save_fig(fig, "binder_crossing")

# 4. chi(T) dan skala puncak
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
for L in (8, 16, 32, 64, 128):
    rs = wolff_by_L[L]
    ax1.plot([r["T"] for r in rs], [r["chi"] for r in rs], "o-", ms=3, label=f"L={L}")
ax1.axvline(TC_EXACT, color="gray", ls=":", lw=1)
ax1.set_xlabel("T"); ax1.set_ylabel("Suseptibilitas chi"); ax1.set_title("chi(T, L)"); ax1.legend(fontsize=8)
ax2.loglog(Ls_fit, chi_peaks, "o", label="Data chi_max")
ax2.loglog(Ls_fit, np.exp(c_chi) * Ls_fit**gamma_nu_hat, "--", label=f"Fit gamma/nu={gamma_nu_hat:.3f}+-{se_gamma_nu:.3f}")
ax2.set_xlabel("L"); ax2.set_ylabel("chi_max"); ax2.set_title("Skala Puncak chi_max ~ L^(gamma/nu)"); ax2.legend(fontsize=8)
save_fig(fig, "chi_scaling")

# 5. tau_int vs L tiga algoritma log-log
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for algo_nm, lbl in [("metropolis", "Metropolis"), ("wolff", "Wolff"), ("sw", "Swendsen-Wang")]:
    taus = Z_DYN[algo_nm]["taus"]
    ax.loglog(Ls_fit, taus, "o-", label=f"{lbl}: z={Z_DYN[algo_nm]['z_hat']:.3f}+-{Z_DYN[algo_nm]['se']:.3f}")
ax.set_xlabel("L"); ax.set_ylabel("tau_int(|m|) di Tc"); ax.set_title("Critical Slowing Down: tau_int vs L"); ax.legend(fontsize=9)
save_fig(fig, "tau_scaling")

# 6. Data collapse (nu=1) dan puncak Cv ~ ln L
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
for L in (16, 32, 64, 128):
    rs = [r for r in wolff_by_L[L] if 2.05 <= r["T"] <= 2.50]
    x_sc = [(r["T"] - TC_EXACT) * (L ** 1.0) for r in rs]
    y_sc = [r["abs_m_mean"] * (L ** 0.125) for r in rs]
    ax1.plot(x_sc, y_sc, "o", ms=3, label=f"L={L}")
ax1.set_xlabel("(T - Tc) L^(1/nu), nu=1"); ax1.set_ylabel("|m| L^(beta/nu), beta/nu=1/8")
ax1.set_title("Data Collapse Magnetisasi"); ax1.legend(fontsize=8)
cv_peaks = [max(r["cv"] for r in wolff_by_L[int(L)]) for L in Ls_fit]
ax2.plot(np.log(Ls_fit), cv_peaks, "s-", label="Cv_max")
ax2.set_xlabel("ln L"); ax2.set_ylabel("Cv_max"); ax2.set_title("Skala Puncak Kalor Jenis Cv ~ ln L"); ax2.legend(fontsize=8)
save_fig(fig, "data_collapse")

# 7. Perbandingan kecepatan 3 bahasa (batang log)
fig, ax = plt.subplots(figsize=(9, 4.5))
labels = list(BENCH_SUMMARY.keys())
vals = [BENCH_SUMMARY[k]["median_ns"] for k in labels]
errs = [BENCH_SUMMARY[k]["mad_ns"] for k in labels]
ax.barh(labels, vals, xerr=errs, color="steelblue", alpha=0.85)
ax.set_xscale("log")
ax.set_xlabel("ns per pembaruan-spin (skala log)"); ax.set_title("Kecepatan Lintas Bahasa & Optimasi (L=32, Tc)")
save_fig(fig, "speed_comparison")

# 8. Tabel paritas sebagai gambar
fig, ax = plt.subplots(figsize=(9, 2.8))
ax.axis("off")
cols = ["Algoritma", "Langkah", "Selisih C-C++", "Selisih C-Py", "Selisih C++-Py", "Hash FNV-1a (64-bit)"]
rows_tbl = [
    [an, "3000", str(PARITY_RESULTS[an]["diff_c_cpp"]), str(PARITY_RESULTS[an]["diff_c_py"]),
     str(PARITY_RESULTS[an]["diff_cpp_py"]), PARITY_RESULTS[an]["py_hash"]]
    for an in ("metropolis", "wolff", "sw")
]
tbl = ax.table(cellText=rows_tbl, colLabels=cols, loc="center", cellLoc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1.1, 1.5)
ax.set_title("Tabel Paritas Bit-Identik Lintas 3 Bahasa (L=8, beta=0.44, Cold Start)", fontsize=11)
save_fig(fig, "parity_table")
print(f"8 pasang grafik PNG+SVG berhasil disimpan ke {FIG_DIR.relative_to(ROOT)}")


8 pasang grafik PNG+SVG berhasil disimpan ke outputs/figures


## 8. Validasi Akhir, Ukuran Kode (`c13`), Reproduksibilitas (`c10`) & Ekspor `results.json`

In [9]:
def verify_figures(fig_map):
    if len(fig_map) != 8:
        return False
    for paths in fig_map.values():
        for rel in paths:
            fp = ROOT / rel
            if not fp.exists() or fp.stat().st_size == 0:
                return False
            if fp.suffix == ".svg":
                ET.parse(fp)
    return True

C12_FIGS_OK = verify_figures(SAVED_FIGS)
assert C12_FIGS_OK, "Pelanggaran c12: grafik tidak lengkap atau SVG tidak valid!"

# Hitung jumlah baris kode sumber (c13)
src_files = [
    PROJ_DIR / "c" / "rng.h", PROJ_DIR / "c" / "ising.h", PROJ_DIR / "c" / "ising.c",
    PROJ_DIR / "c" / "main.c", PROJ_DIR / "c" / "test_c.c",
    PROJ_DIR / "cpp" / "rng.hpp", PROJ_DIR / "cpp" / "ising.hpp", PROJ_DIR / "cpp" / "ising.cpp",
    PROJ_DIR / "cpp" / "main.cpp", PROJ_DIR / "cpp" / "test_cpp.cpp",
]
CODE_LINES = {str(p.relative_to(ROOT)): len(p.read_text(encoding="utf-8").splitlines()) for p in src_files}
C13_LINES_OK = all(n <= 600 for n in CODE_LINES.values())
assert C13_LINES_OK, f"Pelanggaran c13: ada berkas > 600 baris: {CODE_LINES}"

# Sidik jari numerik untuk cek 2x eksekusi (c10)
fp_payload = json.dumps({
    "parity": PARITY_RESULTS,
    "max_enum_rel_err": round(max_enum_rel_err, 15),
    "gamma_nu": round(float(gamma_nu_hat), 8),
    "beta_nu": round(float(beta_nu_hat), 8),
    "z_dyn": Z_DYN,
    "py_sim": [(r["L"], round(r["T"], 6), round(r["e_mean"], 8)) for r in PY_SIM_ROWS],
}, sort_keys=True)
NUMERICAL_SHA256 = hashlib.sha256(fp_payload.encode("utf-8")).hexdigest()

res_path = OUT_DIR / "results.json"
prev_sha = None
if res_path.exists():
    try:
        prev_sha = json.loads(res_path.read_text(encoding="utf-8")).get("reproducibility", {}).get("numerical_sha256")
    except Exception:
        prev_sha = None
two_runs_identical = (prev_sha is not None) and (prev_sha == NUMERICAL_SHA256)

c4_ok = all(v["within_4se"] for v in C4_ONSAGER.values())
c5_ok = all(v["within_4se"] for v in C5_YANG.values())
c6_ok = all(BINDER_CROSSINGS[k]["within_0.01"] for k in ("16_32", "32_64", "64_128"))
c7_ok = abs(gamma_nu_hat - 1.75) <= 0.10
c8_ok = abs(beta_nu_hat - 0.125) <= 0.03
c9_ok = (Z_DYN["wolff"]["z_hat"] < Z_DYN["metropolis"]["z_hat"] - 1.0) and (1.8 <= Z_DYN["metropolis"]["z_hat"] <= 2.5)

RESULTS_DOC = {
    "session": "day008a",
    "date": "2026-11-26",
    "master_seed": MASTER_SEED,
    "run_completed_without_errors": True,
    "environment": {
        "python": platform.python_version(),
        "arch": platform.machine(),
        "cpu_cores": os.cpu_count(),
        "cc": "Apple clang 16.0.0 (-std=c11 -O3 -fno-fast-math -ffp-contract=off)",
        "cxx": "Apple clang 16.0.0 (-std=c++20 -O3 -fno-fast-math -ffp-contract=off)",
    },
    "workload": {
        "c_cpp": "L in {8,16,32,64,128}, 36 T in [1.8, 3.4]; Metropolis 2^18 (L<=64) / 2^16 (L=128); Wolff 2^17 (L<=64) / 2^15 (L=128); SW 2^17 (L<=64) / 2^14 (L=128)",
        "python_notebook": "Wolff only: L in {8,16,32} (36 T, 600/500/300 meas), L=64 (12 T rapat, 150 meas), L=128 tidak dijalankan di notebook",
        "python_sim_sec": round(T_PY_SIM_SEC, 2),
        "total_notebook_sec": round(time.perf_counter() - T_NB_START, 2),
    },
    "reproducibility": {
        "numerical_sha256": NUMERICAL_SHA256,
        "two_runs_identical": two_runs_identical,
    },
    "parity": PARITY_RESULTS,
    "exact_anchors": {
        "tc_exact": TC_EXACT,
        "e_tc_extrapolated_invL2": round(float(intercept_e_tc), 6),
        "e_tc_exact": round(E_TC_EXACT, 6),
        "onsager": C4_ONSAGER,
        "yang": C5_YANG,
    },
    "fss": {
        "binder_crossings": BINDER_CROSSINGS,
        "gamma_over_nu": {"hat": round(float(gamma_nu_hat), 4), "se": round(se_gamma_nu, 4)},
        "beta_over_nu": {"hat": round(float(beta_nu_hat), 4), "se": round(se_beta_nu, 4)},
        "dynamic_exponents_z": Z_DYN,
    },
    "speed_ns_per_spin_update": BENCH_SUMMARY,
    "code_lines": CODE_LINES,
    "figures": SAVED_FIGS,
    "criteria": {
        "c1_parity_bit_identical": {"status": "tercapai" if all_parity_ok else "belum tercapai", "measured": "0 selisih di 3 algoritma & 3 bahasa"},
        "c2_exact_enumeration": {"status": "tercapai", "measured": {"max_rel_err": float(f"{max_enum_rel_err:.2e}"), "l4_sim_z": round(z_l4_sim, 3)}},
        "c3_thread_invariance": {"status": "tercapai" if C3_THREADS_OK else "belum tercapai", "measured": "identik untuk 1, 2, 4, 8 thread di C dan C++"},
        "c4_onsager_energy": {"status": "tercapai" if c4_ok else "sebagian", "measured": C4_ONSAGER},
        "c5_yang_magnetization": {"status": "tercapai" if c5_ok else "sebagian", "measured": C5_YANG},
        "c6_binder_crossing": {"status": "tercapai" if c6_ok else "sebagian", "measured": BINDER_CROSSINGS},
        "c7_gamma_over_nu": {"status": "tercapai" if c7_ok else "belum tercapai", "measured": round(float(gamma_nu_hat), 4)},
        "c8_beta_over_nu": {"status": "tercapai" if c8_ok else "belum tercapai", "measured": round(float(beta_nu_hat), 4)},
        "c9_dynamic_exponent": {"status": "tercapai" if c9_ok else "belum tercapai", "measured": {k: v["z_hat"] for k, v in Z_DYN.items()}},
        "c10_notebook_reproducible": {"status": "tercapai" if two_runs_identical else "sebagian", "measured": {"sha256_prefix": NUMERICAL_SHA256[:16], "two_runs_identical": two_runs_identical}},
        "c11_all_tests_pass": {"status": "tercapai", "measured": "make test C/C++ dan seluruh assert notebook lulus"},
        "c12_figures_valid": {"status": "tercapai" if C12_FIGS_OK else "belum tercapai", "measured": len(SAVED_FIGS)},
        "c13_code_size_reported": {"status": "tercapai" if C13_LINES_OK else "belum tercapai", "measured": max(CODE_LINES.values())},
        "c14_speed_comparison": {"status": "tercapai", "measured": {k: v["median_ns"] for k, v in BENCH_SUMMARY.items()}},
    },
}

txt = json.dumps(RESULTS_DOC, indent=2) + "\n"
for bad in ("/" + "Users/", "/" + "home/", "/" + "tmp", "file" + "://"):
    assert bad not in txt, f"Path terlarang: {bad}"
res_path.write_text(txt, encoding="utf-8")
print(f"=== Ringkasan Kriteria day008a (SHA256={NUMERICAL_SHA256[:16]}, 2x={two_runs_identical}) ===")
for ck, cv in RESULTS_DOC["criteria"].items():
    print(f"  {ck:28s}: {cv['status']}")


=== Ringkasan Kriteria day008a (SHA256=551871e2fc28ee31, 2x=True) ===
  c1_parity_bit_identical     : tercapai
  c2_exact_enumeration        : tercapai
  c3_thread_invariance        : tercapai
  c4_onsager_energy           : tercapai
  c5_yang_magnetization       : tercapai
  c6_binder_crossing          : tercapai
  c7_gamma_over_nu            : tercapai
  c8_beta_over_nu             : tercapai
  c9_dynamic_exponent         : tercapai
  c10_notebook_reproducible   : tercapai
  c11_all_tests_pass          : tercapai
  c12_figures_valid           : tercapai
  c13_code_size_reported      : tercapai
  c14_speed_comparison        : tercapai
